# UERC26 Ear Recognition Training Pipeline (Colab L4 Optimized)
This notebook trains a Swin V2 base backbone with an AdaFace head for ear recognition.
It is highly optimized for Google Colab with an L4 GPU (12GB+ RAM).

**Optimizations:**
- Reads `.webp` files dynamically from the CSV paths (lazy-loading).
- Performs standalone CLAHE and Unsharp masking on CPU parallel workers.
- Uses `num_workers=4` and `prefetch_factor=2` to ensure GPU is never starved.
- Swin V2 base backbone.
- Warmup + cosine LR schedule for stability and final accuracy.
- Pairwise eval aligns with verification metrics (VER/AUC/EER).


In [ ]:
# 1. Mount Google Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

# Ensure directories exist
!mkdir -p /content/checkpoints

# Unzip the dataset (Assumes your ZIP contains the image dataset structured identically)
# Update the path to your dataset zip in Drive
zip_path = '/content/drive/MyDrive/UERC26_modified_new.zip'
import os
if not os.path.exists('/content/UERC26_modified_new/data/public/dataset_split.csv'):
    print("Unzipping dataset...")
    !unzip -q "{zip_path}" -d /content/
else:
    print("Dataset already unzipped.")


Mounted at /content/drive
Unzipping dataset...


In [ ]:
# 2. Import Modules
import os
import csv
import time
import random
import numpy as np
from PIL import Image
import cv2
from pathlib import Path

# VERY IMPORTANT FOR DATALOADER PERFORMANCE ON COLAB:
cv2.setNumThreads(0)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.v2 as v2
import torchvision.models as models
from tqdm.notebook import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# 3. CPU Preprocessing Utilities (CLAHE & Sharpening)
def apply_clahe(image_np, clip_limit=2.0, tile_grid_size=(8, 8)):
    # Applies CLAHE in LAB color space
    lab = cv2.cvtColor(image_np, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def apply_sharpening(image_np, sigma=1.0, amount=0.5):
    # Unsharp masking
    blurred = cv2.GaussianBlur(image_np, (0, 0), sigma)
    sharpened = cv2.addWeighted(image_np, 1.0 + amount, blurred, -amount, 0)
    return np.clip(sharpened, 0, 255).astype(np.uint8)


In [ ]:
# 4. Optimized Dataset Classes
class EarIdentificationDataset(Dataset):
    """
    Optimized for lazy-loading WebP images on Colab.
    Applies CLAHE and Sharpening on CPU workers.
    """
    def __init__(self, split, root_dir, split_csv='dataset_split.csv',
                 classes_mapping=None, is_train=True):
        self.root_dir = root_dir
        self.split = split
        self.is_train = is_train

        self.images = []
        split_path = os.path.join(root_dir, split_csv)

        with open(split_path, 'r') as f:
            reader = csv.reader(f, delimiter='\t')
            for row in reader:
                if len(row) >= 2 and row[1] == split:
                    # Replace .jpg/.png with .webp
                    path_str = row[0].rsplit('.', 1)[0] + '.webp'
                    self.images.append(path_str)

        subjects = sorted(set(os.path.dirname(img) for img in self.images))
        if classes_mapping is not None:
            self.classes_mapping = classes_mapping
        else:
            self.classes_mapping = {s: i for i, s in enumerate(subjects)}
        self.num_classes = len(self.classes_mapping)

        # Filter to existing subjects
        self.images = [img for img in self.images if os.path.dirname(img) in self.classes_mapping]
        self.labels = [self.classes_mapping[os.path.dirname(img)] for img in self.images]

        self.transform = transforms.Compose([transforms.ToTensor()])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = str(Path(self.root_dir) / img_name)

        # Lazy loading strictly from disk
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Cannot read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # CPU Augmentations (Only training)
        if self.is_train:
            if random.random() < 0.5:
                image = apply_clahe(image)
            if random.random() < 0.3:
                image = apply_sharpening(image)

        # To Tensor
        image = Image.fromarray(image)
        image = self.transform(image)
        return image, self.labels[idx]

class EarPairwiseDataset(Dataset):
    """
    Evaluation Dataset. Assumes inputs are already 224x224.
    """
    def __init__(self, root_dir, pairs_csv):
        self.root_dir = root_dir
        self.pairs = []
        self.labels = []

        pairs_path = os.path.join(root_dir, pairs_csv)
        with open(pairs_path, 'r') as f:
            reader = csv.reader(f, delimiter='\t')
            for row in reader:
                if len(row) >= 3:
                    # Dynamic WebP replacement
                    p1 = row[0].rsplit('.', 1)[0] + '.webp'
                    p2 = row[1].rsplit('.', 1)[0] + '.webp'
                    self.pairs.append((p1, p2))
                    self.labels.append(int(row[2]))

        # Inputs are already 224x224. Just ToTensor and Normalize.
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_path = str(Path(self.root_dir) / self.pairs[idx][0])
        img2_path = str(Path(self.root_dir) / self.pairs[idx][1])

        img1 = cv2.cvtColor(cv2.imread(img1_path), cv2.COLOR_BGR2RGB)
        img2 = cv2.cvtColor(cv2.imread(img2_path), cv2.COLOR_BGR2RGB)

        img1 = self.transform(Image.fromarray(img1))
        img2 = self.transform(Image.fromarray(img2))
        return img1, img2, self.labels[idx]


In [ ]:
# 5. Model: Swin V2 Base & AdaFace
class SwinV2EarModel(nn.Module):
    def __init__(self, embedding_dim=512, freeze_backbone=False):
        super().__init__()

        self.backbone = models.swin_v2_b(weights=models.Swin_V2_B_Weights.DEFAULT)
        self.backbone.head = nn.Identity()
        self.backbone_dim = 1024

        # Logic to freeze or unfreeze weights based on your parameter
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        else:
            for param in self.backbone.parameters():
                param.requires_grad = True # Fully unfrozen for fine-tuning

        self.projection = nn.Sequential(
            nn.Linear(self.backbone_dim, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
        )

    def forward(self, x, return_norms=False):
        # FIX: Dynamically resize 224x224 to 256x256 before passing to SwinV2
        if x.shape[-1] != 256 or x.shape[-2] != 256:
            x = F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)

        features = self.backbone(x)
        embeddings = self.projection(features)
        norms = torch.norm(embeddings, p=2, dim=1)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return (embeddings, norms) if return_norms else embeddings

class AdaFaceHead(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.25, h=0.333, t_alpha=0.01):
        super().__init__()
        self.s, self.m, self.h, self.t_alpha = s, m, h, t_alpha
        self.out_features = out_features
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.register_buffer('batch_mean', torch.ones(1) * 20.0)
        self.register_buffer('batch_std', torch.ones(1) * 100.0)

    def forward(self, embeddings, norms, labels):
        with torch.no_grad():
            self.batch_mean = (1 - self.t_alpha) * self.batch_mean + self.t_alpha * norms.mean()
            self.batch_std  = (1 - self.t_alpha) * self.batch_std  + self.t_alpha * (norms.std() + 1e-6)

        margin_scaler = (norms - self.batch_mean) / (self.batch_std + 1e-3)
        margin_scaler = torch.clamp(self.h * margin_scaler, -1.0, 1.0)

        cos_theta = F.linear(F.normalize(embeddings), F.normalize(self.weight)).clamp(-1.0+1e-7, 1.0-1e-7)
        theta = torch.acos(cos_theta)
        one_hot = F.one_hot(labels, self.out_features).float()

        g_add = self.m * (1.0 + margin_scaler)
        target_logits = torch.cos(theta + self.m) - g_add.unsqueeze(1)
        output = (cos_theta * (1.0 - one_hot) + target_logits * one_hot) * self.s
        return F.cross_entropy(output, labels)

In [ ]:
# 6. Evaluation Function & Metrics
def compute_metrics(labels, scores):
    from scipy.interpolate import interp1d
    from sklearn.metrics import roc_curve, roc_auc_score
    fpr, tpr, _ = roc_curve(labels, scores)
    order = np.argsort(fpr)
    fpr, tpr = fpr[order], tpr[order]
    interp = interp1d(fpr, tpr, bounds_error=False, fill_value=(0.0, 1.0))
    metrics = {}
    metrics["VER@0.1%"] = float(interp(0.001))
    metrics["VER@1%"] = float(interp(0.01))
    fnr = 1 - tpr
    # Interpolated EER: find exact crossing of FPR == FNR
    abs_diff = np.abs(fnr - fpr)
    idx = np.argmin(abs_diff)
    if idx > 0 and abs_diff[idx - 1] < abs_diff[idx]:
        idx = idx - 1
    # Linear interpolation between the two closest points
    if idx < len(fpr) - 1:
        x0, x1 = fpr[idx], fpr[idx + 1]
        y0, y1 = fnr[idx], fnr[idx + 1]
        if (x1 - x0 + y0 - y1) != 0:
            eer = (x0 + y0) / 2  # fallback
            t = (y0 - x0) / ((x1 - x0) + (y0 - y1) + 1e-12)
            eer = x0 + t * (x1 - x0)
        else:
            eer = (fpr[idx] + fnr[idx]) / 2
    else:
        eer = (fpr[idx] + fnr[idx]) / 2
    metrics["EER"] = float(eer)
    metrics["AUC"] = float(roc_auc_score(labels, scores))
    return metrics

def evaluate_verification(model, dataloader, device):
    model.eval()
    all_scores, all_labels = [], []
    with torch.no_grad():
        for img1, img2, labels in tqdm(dataloader, desc="Verif eval", leave=False):
            img1 = img1.to(device, non_blocking=True, memory_format=torch.channels_last)
            img2 = img2.to(device, non_blocking=True, memory_format=torch.channels_last)
            sim = F.cosine_similarity(model(img1), model(img2))
            all_scores.extend(sim.cpu().numpy())
            all_labels.extend(labels.numpy())

    return compute_metrics(np.array(all_labels), np.array(all_scores))


In [ ]:
# 7. Setup Dataloaders and Model
DATA_ROOT = '/content/UERC26_modifies_new/data/public' # Change if your structure differs
BATCH_SIZE = 192
NUM_WORKERS = 8 # A100 typically has more host CPU to feed the GPU
EPOCHS = 25

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

print("Loading Datasets...")
train_dataset = EarIdentificationDataset('train', DATA_ROOT)
val_pair_dataset = EarPairwiseDataset(DATA_ROOT, 'pairs_val.csv') if os.path.exists(os.path.join(DATA_ROOT, 'pairs_val.csv')) else None

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=True, prefetch_factor=4 # Prefetch factor keeps GPU fed
)

if val_pair_dataset:
    val_loader = DataLoader(
        val_pair_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4
    )

print(f"Train samples: {len(train_dataset)}")

# FIXED: Set freeze_backbone to False for full fine-tuning
model = SwinV2EarModel(embedding_dim=512, freeze_backbone=False).to(device, memory_format=torch.channels_last)
loss_head = AdaFaceHead(in_features=512, out_features=train_dataset.num_classes).to(device)
scaler = torch.amp.GradScaler("cuda")

Loading Datasets...
Train samples: 179172
Downloading: "https://download.pytorch.org/models/swin_v2_b-781e5279.pth" to /root/.cache/torch/hub/checkpoints/swin_v2_b-781e5279.pth


100%|██████████| 336M/336M [00:01<00:00, 230MB/s]


In [ ]:
# 8. Dry Run to check DataLoader speed
print("Checking DataLoader performance...")
data_times, gpu_times = [], []
end = time.time()
for i, (imgs, lbls) in enumerate(train_loader):
    data_times.append(time.time() - end)
    t = time.time()
    imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
    gpu_times.append(time.time() - t)
    end = time.time()
    if i >= 19: break
print(f"Avg data load: {sum(data_times)/len(data_times)*1000:.1f}ms | Avg GPU transfer: {sum(gpu_times)/len(gpu_times)*1000:.1f}ms")
print("Target: data load should be << 50ms per batch. If higher, adjust num_workers.")


Checking DataLoader performance...
Avg data load: 170.3ms | Avg GPU transfer: 0.2ms
Target: data load should be << 50ms per batch. If higher, adjust num_workers.


In [ ]:
# 9. Setup & Training Loop (15 Epochs, Fully Unfrozen, Smooth LR)
EPOCHS = 15
EVAL_INTERVAL = 1 # Evaluate every epoch so we can watch it learn

# Initialize fully unfrozen model
model = SwinV2EarModel().to(device, memory_format=torch.channels_last)
loss_head = AdaFaceHead(in_features=512, out_features=train_dataset.num_classes).to(device)
scaler = torch.amp.GradScaler("cuda")

# GPU Transforms
gpu_transform = v2.Compose([
    v2.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    v2.RandomGrayscale(p=0.2),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Give the whole model a fighting chance from Epoch 1
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-4}, # Healthy LR for SwinV2
    {'params': model.projection.parameters(), 'lr': 5e-4}, # Slightly higher for new layers
    {'params': loss_head.parameters(), 'lr': 5e-4},
], weight_decay=1e-4)

# Smooth decay down to 1e-6 over the 15 epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_auc = 0.0
patience_limit = 4
patience_counter = 0

print("Starting 15-Epoch Unfrozen Training...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_head.train()
    total_loss, num_batches = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)
        images = gpu_transform(images)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            embeddings, norms = model(images, return_norms=True)
            loss = loss_head(embeddings, norms, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        # Clip gradients to prevent exploding logits from AdaFace
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        torch.nn.utils.clip_grad_norm_(loss_head.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    print(f"Epoch {epoch} | Avg Loss: {total_loss/max(num_batches, 1):.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    # Evaluation
    if val_pair_dataset and epoch % EVAL_INTERVAL == 0:
        metrics = evaluate_verification(model, val_loader, device)
        print(f"VAL | VER@0.1%: {metrics['VER@0.1%']*100:.2f}% | VER@1%: {metrics['VER@1%']*100:.2f}% | EER: {metrics['EER']*100:.2f}% | AUC: {metrics['AUC']*100:.2f}%")

        if metrics['AUC'] > best_auc:
            best_auc = metrics['AUC']
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'loss_head_state_dict': loss_head.state_dict(),
                'val_auc': best_auc,
                'embedding_dim': 512,
                'backbone': 'swin_v2_b',
            }, '/content/checkpoints/best_model.pt')
            print("★ NEW BEST MODEL SAVED!")
        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{patience_limit}")
            if patience_counter >= patience_limit:
                print(f"[!] Early stopping triggered at epoch {epoch}")
                break

    scheduler.step()

print(f"Training Complete! Best AUC: {best_auc*100:.2f}%")

Starting 15-Epoch Unfrozen Training...


Epoch 1/15:   0%|          | 0/933 [00:00<?, ?it/s]

Epoch 1 | Avg Loss: 35.5497 | LR: 1.00e-04


Verif eval:   0%|          | 0/113 [00:00<?, ?it/s]

VAL | VER@0.1%: 10.39% | VER@1%: 35.85% | EER: 16.11% | AUC: 91.71%
★ NEW BEST MODEL SAVED!


...

VAL | VER@0.1%: 78.20% | VER@1%: 90.44% | EER: 4.54% | AUC: 98.83%
No improvement. Patience: 1/4


Epoch 15/15:   0%|          | 0/933 [00:00<?, ?it/s]

Epoch 15 | Avg Loss: 6.9051 | LR: 2.08e-06


Verif eval:   0%|          | 0/113 [00:00<?, ?it/s]

VAL | VER@0.1%: 79.18% | VER@1%: 90.55% | EER: 4.45% | AUC: 98.85%
★ NEW BEST MODEL SAVED!
Training Complete! Best AUC: 98.85%


In [ ]:
# 9b. Final test evaluation after training
# Loads the best checkpoint and evaluates on pairs_test.csv from the public folder.

TEST_EVAL = True
if TEST_EVAL and os.path.exists(os.path.join(DATA_ROOT, 'pairs_test.csv')):
    print("Loading best model for final test evaluation...")
    best_path = '/content/checkpoints/best_model.pt'
    if os.path.exists(best_path):
        checkpoint = torch.load(best_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        loss_head.load_state_dict(checkpoint['loss_head_state_dict'])
    else:
        print(f"Best checkpoint not found at {best_path}. Skipping test eval.")
        checkpoint = None

    if checkpoint is not None:
        test_pair_dataset = EarPairwiseDataset(DATA_ROOT, 'pairs_test.csv')
        test_loader = DataLoader(
            test_pair_dataset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=2
        )
        test_metrics = evaluate_verification(model, test_loader, device)
        print(
            "TEST | VER@0.1%: {ver01:.2f}% | VER@1%: {ver1:.2f}% | EER: {eer:.2f}% | AUC: {auc:.2f}%".format(
                ver01=test_metrics['VER@0.1%']*100,
                ver1=test_metrics['VER@1%']*100,
                eer=test_metrics['EER']*100,
                auc=test_metrics['AUC']*100,
            )
        )
else:
    print("pairs_test.csv not found or TEST_EVAL disabled. Skipping test eval.")


In [ ]:
# 10. Submission Generator (Embeddings Extraction)
class EarImageDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.images = []
        for dirpath, _, filenames in os.walk(root_dir):
            for fname in sorted(filenames):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                    rel_path = os.path.relpath(os.path.join(dirpath, fname), root_dir)
                    self.images.append(rel_path.replace('\\', '/'))
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img_name = self.images[idx]
        raw = cv2.imread(os.path.join(self.root_dir, img_name))
        if raw is None:
            raise FileNotFoundError(f"Cannot read image: {os.path.join(self.root_dir, img_name)}")
        image = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
        image = apply_clahe(image)  # Apply CLAHE at inference for consistency
        return self.transform(Image.fromarray(image)), img_name

def extract_embeddings(model, dataloader, device):
    model.eval()
    features = {}
    with torch.no_grad():
        for images, img_names in tqdm(dataloader, desc="Extracting", leave=False):
            emb = model(images.to(device)).cpu().numpy()
            for name, feat in zip(img_names, emb):
                features[name] = feat
    return features

# Set to True when you want to generate submission files!
GENERATE_SUBMISSION = True

if GENERATE_SUBMISSION:
    import shutil
    print("Loading Best Model...")
    checkpoint = torch.load('/content/checkpoints/best_model.pt')
    model.load_state_dict(checkpoint['model_state_dict'])

    seq_root = '/content/UERC26_modified_new/data/sequestered_anonymized'
    out_dir = '/content/checkpoints/swinv2_solution'
    os.makedirs(out_dir, exist_ok=True)

    if os.path.exists(seq_root):
        print(f"Extracting features for {seq_root}...")
        seq_dataset = EarImageDataset(seq_root)
        seq_loader = DataLoader(seq_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        seq_features = extract_embeddings(model, seq_loader, device)
        npy_path = os.path.join(out_dir, 'sequestered_features.npy')
        np.save(npy_path, seq_features)
        print(f"Saved {len(seq_features)} embeddings to {npy_path}")

        # Push to Google Drive
        drive_out = '/content/drive/MyDrive/UERC26_FInal_Teacher_Output_Swin_Singe_Phase'
        os.makedirs(drive_out, exist_ok=True)
        shutil.copy(npy_path, drive_out)
        shutil.copy('/content/checkpoints/best_model.pt', drive_out)
        print(f"SUCCESS: Saved sequestered_features.npy and best_model.pt to {drive_out}")
    else:
        print(f"Directory {seq_root} not found! Upload the sequestered dataset to generate features.")


NameError: name 'Dataset' is not defined

In [ ]:
# 10. Submission Generator (Embeddings Extraction)
class EarImageDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.images = []
        for dirpath, _, filenames in os.walk(root_dir):
            for fname in sorted(filenames):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                    rel_path = os.path.relpath(os.path.join(dirpath, fname), root_dir)
                    self.images.append(rel_path.replace('\\', '/'))
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img_name = self.images[idx]
        raw = cv2.imread(os.path.join(self.root_dir, img_name))
        if raw is None:
            raise FileNotFoundError(f"Cannot read image: {os.path.join(self.root_dir, img_name)}")
        image = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
        image = apply_clahe(image)  # Apply CLAHE at inference for consistency
        return self.transform(Image.fromarray(image)), img_name

def extract_embeddings(model, dataloader, device):
    model.eval()
    features = {}
    with torch.no_grad():
        for images, img_names in tqdm(dataloader, desc="Extracting", leave=False):
            emb = model(images.to(device)).cpu().numpy()
            for name, feat in zip(img_names, emb):
                features[name] = feat
    return features

# Set to True when you want to generate submission files!
GENERATE_SUBMISSION = True

if GENERATE_SUBMISSION:
    import shutil
    print("Loading Best Model...")
    checkpoint = torch.load('/content/checkpoints/best_model.pt')
    model.load_state_dict(checkpoint['model_state_dict'])

    seq_root = '/content/UERC26_modified_new/data/sequestered_anonymized'
    out_dir = '/content/checkpoints/swinv2_solution'
    os.makedirs(out_dir, exist_ok=True)

    if os.path.exists(seq_root):
        print(f"Extracting features for {seq_root}...")
        seq_dataset = EarImageDataset(seq_root)
        seq_loader = DataLoader(seq_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        seq_features = extract_embeddings(model, seq_loader, device)
        npy_path = os.path.join(out_dir, 'sequestered_features.npy')
        np.save(npy_path, seq_features)
        print(f"Saved {len(seq_features)} embeddings to {npy_path}")

        # Push to Google Drive
        drive_out = '/content/drive/MyDrive/UERC26_Output_Swin_Singe_Phase'
        os.makedirs(drive_out, exist_ok=True)
        shutil.copy(npy_path, drive_out)
        shutil.copy('/content/checkpoints/best_model.pt', drive_out)
        print(f"SUCCESS: Saved sequestered_features.npy and best_model.pt to {drive_out}")
    else:
        print(f"Directory {seq_root} not found! Upload the sequestered dataset to generate features.")
